In [ ]:
(display-table (tastytrade-option-chain api-key "BRK/B") )

In [ ]:
(define a #(1 2 3 2 1))

In [ ]:
(define b #(1 2 3 4 5))

In [ ]:
(< a b)

In [ ]:
(define c (< a b))

In [ ]:
(define d (= a b))

In [ ]:
d

In [ ]:
(vector-select a d)

In [ ]:
(vector-select b d)

In [ ]:
(vector-select a c)

In [ ]:
(vector-select b c)

(define dd (tastytrade-option-chain api-key "BRK/B"))

In [ ]:
(length dd)

In [ ]:
(define dd (tastytrade-option-chain api-key "BRK/B"))

In [ ]:
(length dd)

In [ ]:
(display-table dd '(("implied-volatility"  ".1%") ("delta" ".1f") ("volume" ",.0f")))

In [ ]:
; stratify_example.lsp
;
; Stratification tables: a big table of loans, cut up into many small
; tables, each with a row per bucket of one column (coupons from 3.0 to 5.0,
; 5.0 to 6.0, ...; ten buckets of loan age; one row per state) and, in each
; row, the loans' count, total balance, share of the pool, and
; balance-weighted averages. See "Stratification tables" in
; lisp_interpreter_reference.md.
;
; The pool here is 5,000 made-up loans, so it runs anywhere. To use real
; data, load the table from SQLite instead, e.g.
;
;   (define loans
;     (with-sqlite (conn "/Users/morris/freddie_data/freddie_sample.db")
;       (sqlite-query conn "SELECT current_actual_upb AS balance, current_interest_rate AS rate,
;                                  loan_age AS age, estimated_ltv AS ltv, orig_dti AS dti
;                           FROM loan_performance
;                           WHERE monthly_reporting_period = '2020-01-01'")))
;
; Run it from the examples directory:
;   python3 ../lisp_interpreter.py stratify_example.lsp

; --- A made-up pool of loans -------------------------------------------------
(random-seed 1)
(define n 5000)
(define (make-column value) (list->vector (loop repeat n collect (value))))
(define states '("CA" "TX" "FL" "NY" "IL" "WA" "AZ" "NC" "GA" "CO"))

(define loans
  (make-table
    "balance"   (make-column (lambda () (round (random-float 40000 800000))))
    "rate"      (make-column (lambda () (/ (round (* 8 (random-float 2.75 8.25))) 8.0)))   ; eighths
    "age"       (make-column (lambda () (random-int 0 120)))
    "ltv"       (make-column (lambda () (round (random-float 40 97))))
    "fico"      (make-column (lambda () (random-int 600 820)))
    "state"     (make-column (lambda () (list-ref states (random-int 0 (- (length states) 1)))))
    "first-pay" (make-column (lambda () (date-add-months (date 2016 1 1) (random-int 0 107))))))

; --- What each small table shows -----------------------------------------------
; Every table has the bucket, the count of loans, their total balance
; ("total balance", since the weight is the balance column), and its share of
; the pool ("percent"); then these, each averaged by balance.
(define summaries
  '(("rate" weighted-mean "WAC")
    ("age"  weighted-mean "WALA")
    ("ltv"  weighted-mean "LTV")
    ("fico" weighted-mean "FICO")))

; How to show the numbers (see display-table).
(define formats
  '(("total balance" ",.0f") ("percent" ".1%") ("WAC" ".3f") ("WALA" ".1f") ("LTV" ".1f") ("FICO" ".0f")))

; --- The tables ----------------------------------------------------------------
(define tables
  (stratify-all loans
    '(("rate"      (3.0 4.0 5.0 6.0 7.0 8.0))     ; coupon ranges
      ("balance"   (100000 200000 300000 400000 500000 600000 700000))
      ("age"       (equal-count 5))              ; five buckets, 1,000 loans each
      ("ltv"       (equal-weight 4))             ; four buckets, a quarter of the balance each
      ("state"     (top 5))                      ; the five biggest states, then the rest
      ("first-pay" year)                         ; by year of first payment
      (("state" (top 3)) ("rate" (5.0 7.0))))    ; two columns at once
    summaries
    :weight "balance"))

(dolist (table tables)
  (display-table table formats)
  (newline))

; A single table, made on its own:
(display-table (stratify loans '("fico" (660 700 740 780)) summaries :weight "balance") formats)


In [ ]:
; stratify_example.lsp
;
; Stratification tables: a big table of loans, cut up into many small
; tables, each with a row per bucket of one column (coupons from 3.0 to 5.0,
; 5.0 to 6.0, ...; ten buckets of loan age; one row per state) and, in each
; row, the loans' count, total balance, share of the pool, and
; balance-weighted averages. See "Stratification tables" in
; lisp_interpreter_reference.md.
;
; The pool here is 5,000 made-up loans, so it runs anywhere. To use real
; data, load the table from SQLite instead, e.g.
;
;   (define loans
;     (with-sqlite (conn "/Users/morris/freddie_data/freddie_sample.db")
;       (sqlite-query conn "SELECT current_actual_upb AS balance, current_interest_rate AS rate,
;                                  loan_age AS age, estimated_ltv AS ltv, orig_dti AS dti
;                           FROM loan_performance
;                           WHERE monthly_reporting_period = '2020-01-01'")))
;
; Run it from the examples directory:
;   python3 ../lisp_interpreter.py stratify_example.lsp

; --- A made-up pool of loans -------------------------------------------------
(random-seed 1)
(define n 5000)
(define (make-column value) (list->vector (loop repeat n collect (value))))
(define states '("CA" "TX" "FL" "NY" "IL" "WA" "AZ" "NC" "GA" "CO"))

(define loans
  (make-table
    "balance"   (make-column (lambda () (round (random-float 40000 800000))))
    "rate"      (make-column (lambda () (/ (round (* 8 (random-float 2.75 8.25))) 8.0)))   ; eighths
    "age"       (make-column (lambda () (random-int 0 120)))
    "ltv"       (make-column (lambda () (round (random-float 40 97))))
    "fico"      (make-column (lambda () (random-int 600 820)))
    "state"     (make-column (lambda () (list-ref states (random-int 0 (- (length states) 1)))))
    "first-pay" (make-column (lambda () (date-add-months (date 2016 1 1) (random-int 0 107))))))

; --- What each small table shows -----------------------------------------------
; Every table has the bucket, the count of loans, their total balance
; ("total balance", since the weight is the balance column), and its share of
; the pool ("percent"); then these, each averaged by balance.
(define summaries
  '(("rate" weighted-mean "WAC")
    ("age"  weighted-mean "WALA")
    ("ltv"  weighted-mean "LTV")
    ("fico" weighted-mean "FICO")))

; How to show the numbers (see display-table).
(define formats
  '(("total balance" ",.0f") ("percent" ".1%") ("WAC" ".3f") ("WALA" ".1f") ("LTV" ".1f") ("FICO" ".0f")))

; --- The tables ----------------------------------------------------------------
(define tables
  (stratify-all loans
    '(("rate"      (3.0 4.0 5.0 6.0 7.0 8.0))     ; coupon ranges
      ("balance"   (100000 200000 300000 400000 500000 600000 700000))
      ("age"       (equal-count 5))              ; five buckets, 1,000 loans each
      ("ltv"       (equal-weight 4))             ; four buckets, a quarter of the balance each
      ("state"     (top 5))                      ; the five biggest states, then the rest
      ("first-pay" year)                         ; by year of first payment
      (("state" (top 3)) ("rate" (5.0 7.0))))    ; two columns at once
    summaries
    :weight "balance"))

(dolist (table tables)
  (display-table table formats)
  (newline))

; A single table, made on its own:
(display-table (stratify loans '("fico" (660 700 740 780)) summaries :weight "balance") formats)


(load "othello_example.lsp") (othello human (alpha-beta-searcher 2 weighted-squares) #t)

In [ ]:
(load "chess.lsp")

In [ ]:
(define dd (tastytrade-option-chain api-key "BRK/B"))

In [ ]:
(length dd)

In [ ]:
(length (cdr (car dd)))

In [ ]:
 (load "vol_smile_example.lsp")

In [ ]:
(display (format "BRK/B: {} options\n\n" (table-row-count dd)))


In [ ]:
(load "vol_smile.lsp")


In [ ]:
(define fit (fit-vol-smiles chain :rate 0.045 :max-vol-spread 0.02))


In [ ]:
(show-vol-smiles fit :count 10)


In [ ]:
(load "sec_example.lsp")


In [ ]:
(load "examples/fdic_example.lsp")

In [ ]:
(map car dd)

In [ ]:
(plot-xy (table-column dd "last-price") (list (table-column dd "days-to-expiration") ( table-column dd "strike")))

In [ ]:
(define prices (vector 10 20 30 40 50))
(define squares (vector-map (lambda (x) (* x x)) prices))
(plot-xy prices (list squares))

In [ ]:
(load 
      "plot_chart_example.lsp")

In [ ]:
(load "bea_example.lsp")

In [ ]:
(bea-names)

In [ ]:
(display-table (bea-names))

In [ ]:
(load "map_example.lsp")


In [ ]:
(load "fred_example.lsp")

In [ ]:
(define v_industry (load-csv "/Users/morris/v_data/load_industry.csv"))

In [ ]:
(define v_ticker (load-csv "/Users/morris/v_data/load_ticker.csv"))

In [ ]:
(map car v_industry)

In [ ]:
(map car v_ticker)

In [ ]:
(define stock_list (table-join v_ticker (table-select v_industry '("ticker" "industry")) '("ticker")))

In [ ]:
(table-row-count stock_list)

In [ ]:
(display-table stock_list)

In [ ]:
(save-variables "ticker_industry_list.json" stock_list)

In [ ]:
(map car ss)

In [ ]:
(length (cdr (car ss)))

In [ ]:
(length (cdr (car v_industry)))

In [ ]:
(length (cdr (car v_ticker)))

In [ ]:
(length (cdr (car stock_list)))

In [ ]:
(define industry_count (table-group-by stock_list '(industry) '((count count))))

In [ ]:
(display-table industry_count)

In [ ]:
(schwab-login creds)

In [ ]:
(define brk_history (schwab-price-history creds "BRK/A"))

In [ ]:
(define prices (table-column brk_history "close"))
(define dates (table-column brk_history "date"))

(define drops (vector-drawdowns dates prices 10))
(define last-date (vector-ref dates (- (vector-length dates) 1)))

; Shade each drop from its high until the index got back to it (or until today).
(define (drop-span row)
  (let ((recovery (row-ref row "recovery-date")))
    (list (row-ref row "peak-date")
          (if (null? recovery) last-date recovery)
          (format "-{:.0f}%" (row-ref row "drop-pct")))))

(plot-chart (list (list "BRK/A" dates prices))
            :shade (map drop-span (table-rows drops))
            :title "BRK/A, with drops of more than 10% shaded"
            :y-format "{:,.0f}" :legend #f :width 11 :height 6)

In [ ]:
 (plot-chart (list
(list "BRK/A Prices" (table-column brk_history "date") (table-column brk_history "close"))))

In [ ]:
(define portfolio (schwab-positions creds))

In [ ]:
(display-table (schwab-accounts creds))

In [ ]:
 (schwab-get creds
    "/marketdata/v1/quotes" '(("symbols" . "AAPL,SPY") ("fields" . "quote,reference,fundamental")))

In [ ]:
(define a _4)

In [ ]:
a

In [ ]:
(keys a)

In [ ]:
(hash-table-keys a)

In [ ]:
(hash-table-ref a 'AAPL)

In [ ]:
(define b (hash-table-ref a 'AAPL))

In [ ]:
(define bk (hash-table-keys b))

In [ ]:
(hash-table-ref a bk)

In [ ]:
(map (lambda x (hash-table-ref b x)) bk)

In [ ]:
bk

In [ ]:
(car bk)

In [ ]:
(hash-table-ref b (car bk))

In [ ]:
(map print bk)

In [ ]:
(map quote bk)

In [ ]:
(length portfolio)


In [ ]:
(map car portfolio)

In [ ]:
(map car stock_list)

In [ ]:
(define stock_list (table-rename-column stock_list "ticker" "symbol"))

In [ ]:
(define portfolio_industries (table-join portfolio stock_list "symbol" 'left))

In [ ]:
(define portfolio_industries (table-drop-columns portfolio_industries '("asset-type" "day-gain" "cusip")))

In [ ]:
(display-table portfolio_industries '(("quantity" ",.0f") ("average-price" "0.2f")("market-value" ",.0f") ("unrealized-gain" ",.0f") ))  

In [ ]:
(defstruct animal (name "unknown") (speak (lambda (self) "...")))
(defstruct (dog (:include animal (speak (lambda (self) "Woof!")))))
(defstruct (cat (:include animal (speak (lambda (self) "Meow!")))))

(define (make-noise a) (call-method animal-speak a))  ; only ever mentions `animal`

 

In [ ]:
 
(make-noise (make-dog))      
 

In [ ]:
(make-noise (make-cat))

In [ ]:
(make-noise (make-animal))

In [ ]:
(define dd (make-dog))

In [ ]:
(define cc (make-cat))

In [ ]:
cc

In [ ]:
(save-variables "foo.json" cc dd :leave-out-procedures #t)

In [ ]:
(table-row-count brk_history)

In [ ]:
(display-table brk_history)

In [ ]:
(load-variables "ticker_industry_list.json")

In [ ]:
(display-table stock_list)

In [ ]:
(define sp500 (load-csv "/Users/morris/morris_lisp/spy_holdings/spy_holdings.csv"))

In [ ]:
(display-table sp500)

In [ ]:
(define sp500_ind (table-join sp500 stock_list 'Ticker 'left))

In [ ]:
(display-table sp500_ind)

In [ ]:
(display-table (table-where sp500_ind "industry"  ()))

In [ ]:
(display-table (table-where sp500_ind "ticker" ()))

In [ ]:
(load "/Users/morris/morris_lisp/lisp_interp/examples/investment_paths_example.lsp")

In [ ]:
; A monthly line over grouped quarterly bars, from one BLS table
(define t (bls-series creds '("unemployment-rate" "productivity" "employment-cost-index")))
(define dates (table-column t "date"))
(plot-chart (list (list "unemployment rate" dates (table-column t "unemployment-rate") :symbol "dot")
                  (list "productivity growth" dates (table-column t "productivity") :bars #t))
            :title "Jobs and productivity" :y-label "percent")

; Stacked bars by category, labeled in dollars, with their total as a dashed line
(define states #("New York" "Texas" "California"))
(plot-chart (list (list "wages" states #(60000 50000 70000) :bars #t :labels #t)
                  (list "other income" states #(15000 12000 20000) :bars #t :labels #t)
                  (list "total" states #(75000 62000 90000) :line "dashed" :symbol "diamond" :color "black"))
            :bars  "grouped" :legend "upper left" :y-format "${:,.0f}")

; Two scales: payrolls in thousands, the unemployment rate in percent
(define jobs (bls-series creds '("nonfarm-payrolls" "unemployment-rate") :start-year 2019))
(plot-chart (list (list "payrolls" (table-column jobs "date") (table-column jobs "nonfarm-payrolls"))
                  (list "unemployment rate" (table-column jobs "date") (table-column jobs "unemployment-rate")
                        :secondary #t))
            :y-label "thousands" :secondary-label "percent" :secondary-min 0)

; An area chart with a recession shaded, a line at 4%, and a note
(plot-chart (list (list "unemployment" (table-column jobs "date") (/ (table-column jobs "unemployment-rate") 100)
                        :fill #t :line #t))
            :shade (list (list (date 2020 2 1) (date 2020 4 30) "recession"))
            :y-lines (list (list 0.04 "4%")) :y-format "{:.0%}"
            :notes (list (list (date 2020 4 1) 0.148 "14.8% in April 2020")) :legend #f)

; A fitted line with a band around it
(define x #(1 2 3 4 5 6))
(define fit #(2.0 3.1 4.0 5.2 5.9 7.1))
(plot-chart (list (list "range" x (- fit 0.8) :fill (+ fit 0.8) :color "gray")
                  (list "fit" x fit :line #t :symbol #t)))

; Fifty states, across, richest at the top; a log scale for prices since 1950
(define states (table-sort (census-profile creds "state:*") "median-household-income" #t))
(plot-chart (list (list "median income" (table-column states "name")
                        (table-column states "median-household-income") :bars #t))
            :horizontal #t :y-ticks 5 :y-format "${:,.0f}" :legend #f)
(define cpi (bls-series creds "cpi-nsa" :start-year 1950 :annual #t))
(plot-chart (list (list "CPI" (table-column cpi "date") (table-column cpi "cpi-nsa"))) :y-log #t)
 

In [ ]:
(load "/Users/morris/morris_lisp/lisp_interp/examples/option_methods_example.lsp")

In [ ]:
(google-login creds)

In [ ]:
(google-sheet creds "s&p 500" () sp500)

In [ ]:
(pretty-print-macro while)

In [ ]:
(define mid_sfr_sa_zip "https://files.zillowstatic.com/research/public_csvs/zhvi/Zip_zhvi_uc_sfr_tier_0.33_0.67_sm_sa_month.csv")

(define mid_condo_sa_ip "https://files.zillowstatic.com/research/public_csvs/zhvi/Zip_zhvi_uc_condo_tier_0.33_0.67_sm_sa_month.csv")

(define mid_all_nsa_metro "https://files.zillowstatic.com/research/public_csvs/zhvi/Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_month.csv")

(define high_all_sa_metro "https://files.zillowstatic.com/research/public_csvs/zhvi/Metro_zhvi_uc_sfrcondo_tier_0.67_1.0_sm_sa_month.csv")

(define low_all_sa_metro "https://files.zillowstatic.com/research/public_csvs/zhvi/Metro_zhvi_uc_sfrcondo_tier_0.0_0.33_sm_sa_month.csv")

(define mid_all_sa_zip "https://files.zillowstatic.com/research/public_csvs/zhvi/Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")


In [ ]:
(define z (http-get-csv mid_all_sa_zip #t 1500 :text '("RegionName")))

In [ ]:
(display-table z)

In [ ]:
(car (car z))

In [ ]:
(car (nth z 10))

In [ ]:
(string->date (car (list-ref z 11)))

In [ ]:
(car (list-ref z 11))

In [ ]:
(define tt (make-table "a" #(1 2 3 4) "b" #(4 5 6 7) "c" #(10 20 30 40)))

In [ ]:
(display-table tt)

In [ ]:
(define uu (table-rename-column tt "b" "z"))

In [ ]:
(display-table uu)

In [ ]:
(display-table tt)

In [ ]:
(table-column tt "b")

In [ ]:
(vector-set! (table-column tt "b") 1 99)

In [ ]:
(display-table tt)

In [ ]:
(display-table uu)

In [ ]:
(date-end-of-month (today))

In [ ]:
(date->string (date-end-of-month (today)))

In [ ]:
(display-table (table-where z "RegionName" "05401"))
              

In [ ]:
(define (get-z-values t key . d_list)
    (let ((z (table-where t "RegionName" key)))
        (map
            (lambda (d) (vector-ref
                            (table-column z (date->string (date-end-of-month d)))
                            0)
              )
            d_list)
      )
  )

                 


In [ ]:
(get-z-values z "05401" (date 2025 1 1) (date 2026 1 1))

In [ ]:
(define (home-price-appreciation zip d1 d2)
  (destructuring-bind (start_val end_val) (get-z-values z zip d1 d2)
                      (/ end_val start_val))
  )


In [ ]:
(home-price-appreciation "05401" (date 2025 1 1) (date 2026 1 1))

In [ ]:
(load "vol_smile.lsp")
(define chain (tastytrade-option-chain creds "SPY" 4 25))
(define fit (fit-vol-smiles chain :rate 0.045))
(show-vol-smiles fit)
;


; one surface for all the expirations, with all five terms
(define surface (fit-vol-smiles chain :rate 0.045 :by-expiration #f))
(display (model-report (cdr (car (vol-smile-fit-models surface)))))

In [ ]:
(define stuff (census-shapes "congressional-district")